# 13. Anomaly detection on network metrics (Week 4)

**Owner:** Rishi (Anomaly detection lead)

**Question:** do the market's network metrics behave unusually *before* the labeled stress episodes?

**Input:** `data/processed/features/connectedness_60d.parquet` from notebook 04 (60-day windows every 5 trading days: average correlation, PC1 share, absorption ratio). If Brandon's `data/processed/network_features.parquet` exists, its network density is added too.

**Method** (code in `src/anomaly.py`, tests in `tests/test_anomaly_alerts.py`):
1. **Trailing z-scores.** Each metric, and the 20-day change in average correlation and absorption, is compared with its previous year of values. The current value is not part of its own baseline. A metric fires at z >= 2 in the stress direction.
2. **Walk-forward Isolation Forest** (Liu, Ting and Zhou, 2008) on the same z-scores. It is first fitted on about three years of history, then refitted once a year on past data only. A row is flagged when its score is above the 95th percentile of the training scores. Isolation Forest also flags unusually *calm* markets, so its flags are only kept when the z-score composite points toward stress.
3. **Alert** = either detector fires.

Nothing here uses data from after the date it is reported on. The stress labels are only used at the end, to see whether the alerts came before episodes.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works from the repo root or from notebooks/. RGBFS_DATA_ROOT is only for test runs on fake data.
CODE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "anomaly.py").exists())
DATA_ROOT = Path(os.environ.get("RGBFS_DATA_ROOT", CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT))

from src import anomaly as an
from src import alerts as al

PROC = DATA_ROOT / "data" / "processed"
FEATURES = PROC / "features"
EPISODES_CSV = DATA_ROOT / "docs" / "stress_periods.csv"
OUT = DATA_ROOT / "notebooks" / "outputs" / "rishi_week4"
OUT.mkdir(parents=True, exist_ok=True)
print("Code:", CODE_ROOT)
print("Data:", DATA_ROOT)

In [ ]:
def shade_episodes(ax, episodes, color="tab:red", alpha=0.12):
    for _, ep in episodes.iterrows():
        ax.axvspan(ep.start, ep.end, color=color, alpha=alpha, lw=0)

## 1. Load the network metrics

In [ ]:
conn = pd.read_parquet(FEATURES / "connectedness_60d.parquet").sort_index()
conn.index = pd.to_datetime(conn.index)
base_cols = ["avg_corr", "pc1_share", "absorption"]

net_path = PROC / "network_features.parquet"
if net_path.exists():
    net = pd.read_parquet(net_path)
    if "date" in net.columns:
        net = net.set_index("date")
    net.index = pd.to_datetime(net.index)
    # Brandon's table is weekly (Fridays). Attach the latest value on or before each window date.
    merged = pd.merge_asof(conn.reset_index().rename(columns={conn.index.name or "index": "date"}).sort_values("date"),
                           net[["network_density"]].sort_index().reset_index().rename(columns={net.index.name or "index": "date"}),
                           on="date", direction="backward", tolerance=pd.Timedelta("7D"))
    conn = merged.set_index("date")
    base_cols.append("network_density")
    print("Added Brandon's network_density")
else:
    print("network_features.parquet not found; using connectedness metrics only")

episodes = al.load_episodes(EPISODES_CSV)
print(conn.shape, conn.index.min().date(), "to", conn.index.max().date())
conn[base_cols].describe().round(3)

## 2. Run both detectors

In [ ]:
res = an.detect(conn, base_cols=base_cols)
print("Rows:", len(res))
print("z-score flag rate:     ", round(res.z_flag.mean(), 3))
print("Isolation Forest rate: ", round(res.if_flag.mean(), 3), "(any direction)")
print("IF, stress direction:  ", round(res.if_flag_stress.mean(), 3))
print("Combined alert rate:   ", round(res.alert.mean(), 3))
print("First z-score:", res.z_composite.first_valid_index().date(), " First IF score:", res.if_score.first_valid_index().date())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
ax = axes[0]
ax.plot(conn.index, conn.avg_corr, lw=1, label="avg_corr (60d)")
hit = res.index[res.alert.eq(1)]
ax.scatter(hit, conn.loc[hit, "avg_corr"], s=10, color="black", label="alert", zorder=3)
ax.set_ylabel("Average correlation"); ax.legend(loc="upper left")
ax = axes[1]
ax.plot(res.index, res.z_composite, lw=1, label="z composite (signed)")
ax.axhline(an.Z_THRESHOLD, ls="--", color="gray", lw=1)
ax.set_ylabel("z"); ax.legend(loc="upper left")
ax = axes[2]
ax.plot(res.index, res.if_score, lw=1, label="Isolation Forest score")
ax.plot(res.index, res.if_threshold, lw=1, ls="--", color="gray", label="threshold (from training rows)")
ax.set_ylabel("score"); ax.legend(loc="upper left")
for a in axes:
    shade_episodes(a, episodes)
axes[0].set_title("Network-metric anomalies (red shading = labeled stress episodes, used only for checking)")
plt.tight_layout()
plt.savefig(OUT / "anomaly_timeline.png", dpi=120)
plt.show()

## 3. Did alerts come before the episodes?

An episode counts as **warned** if the first alert is in the 60 trading days before its start (and after the previous episode ended). **Late** means the first alert came in the first 10 days after the start. Both start dates from the stress labels are checked: `start` (centered volatility, the early bound) and `start_trailing` (the late bound).

Alerts are moved onto the trading-day calendar first. A window value dated *t* is used from *t* until the next window arrives (5 trading days later).

In [ ]:
calendar = pd.read_parquet(PROC / "daily_panel.parquet").index
calendar = pd.DatetimeIndex(calendar).sort_values()
daily = an.to_daily(res.drop(columns=["fit_end"]), calendar)
signals = {"combined": daily.alert, "z-score only": daily.z_flag, "IF (stress direction)": daily.if_flag_stress}
summary = pd.DataFrame([al.summarize_signal(name, s, episodes, start_col=col)
                        for col in ["start", "start_trailing"] for name, s in signals.items()])
summary.round(3)

In [ ]:
lt = al.lead_times(daily.alert, episodes)
lt[["episode_id", "name", "start", "first_alert", "lead_days", "already_on", "result"]]

## 4. Sensitivity to the two thresholds

A lower bar warns earlier but raises more false alarms. This grid shows the trade-off so the threshold is chosen on purpose, not by default. **Do not pick the best row and report it as the result:** that would be tuning on the same 13 episodes we evaluate on. Report the default (z = 2, 95th percentile) and show this table as a robustness check.

In [ ]:
rows = []
for z_thr in [1.5, 2.0, 2.5]:
    for q in [0.90, 0.95, 0.99]:
        r = an.detect(conn, base_cols=base_cols, z_threshold=z_thr, if_kwargs={"quantile": q})
        d = an.to_daily(r[["alert"]], calendar)
        s = al.summarize_signal(f"z>={z_thr}, IF q={q}", d.alert, episodes)
        rows.append({"z_threshold": z_thr, "if_quantile": q, **{k: s[k] for k in
                     ["warned_early", "late", "missed", "median_lead_days", "false_alarms_per_year", "day_precision", "day_recall"]}})
sens = pd.DataFrame(rows)
sens.round(3)

## 5. Save outputs

* `data/processed/features/anomaly_network_daily.parquet`: daily anomaly scores and flags for modeling (Week 5) and the dashboard (Week 7). Large, so it goes in the shared Drive `data/processed/features/` folder, not GitHub.
* `notebooks/outputs/rishi_week4/`: small summary tables and the chart.

In [ ]:
daily_out = daily.drop(columns=[c for c in ["if_threshold"] if c in daily.columns])
daily_out.to_parquet(FEATURES / "anomaly_network_daily.parquet")
res.to_csv(OUT / "anomaly_scores_by_window.csv")
summary.to_csv(OUT / "signal_summary.csv", index=False)
lt.to_csv(OUT / "lead_times_combined.csv", index=False)
sens.to_csv(OUT / "threshold_sensitivity.csv", index=False)
al.score_events(daily.alert, episodes).to_csv(OUT / "alert_events.csv", index=False)
print("Saved", FEATURES / "anomaly_network_daily.parquet", daily_out.shape)

## 6. Findings

*To fill in after the real-data run. Things to report:*
* How many of the scored episodes were warned early, late or missed (Section 3), and the median lead.
* False alarms per year and which years they cluster in.
* Episodes before about 2008 only have the z-score detector (Isolation Forest needs three years of history first), and the first z-scores need about a year of windows.
* Whether the result holds across the threshold grid (Section 4).